In [0]:
# =============================================================
# view_pipeline.py
# Dedicated pipeline for LOAD_TYPE = VIEW
# Triggered separately when a GROUP_ID has VIEW entries
# in l1_detail or l2_detail.
#
# What it does:
#   1. Reads l1_detail / l2_detail for this GROUP_ID
#   2. For every row where LOAD_TYPE = 'VIEW':
#      - Strips any leading CREATE OR REPLACE VIEW ... AS
#        if TRANSFORMATION_QUERY already has it (your CSV does)
#      - Extracts the pure SELECT and runs CREATE OR REPLACE VIEW
#   3. Audits result to audit_log
# =============================================================

In [0]:
# ── Parameters ────────────────────────────────────────────────
# DLT pipelines pass config via spark.conf (pipeline configuration),
# not dbutils.widgets. We try spark.conf first, then widgets fallback
# so this notebook works both as a DLT pipeline AND as a manual run.

def get_param(key, default=""):
    """
    Read from spark.conf first (DLT pipeline configuration),
    fall back to dbutils.widgets (manual / job run).
    """
    # Try spark.conf — DLT sets these from pipeline.configuration
    try:
        val = spark.conf.get(f"pipelines.{key}", "")
        if not val:
            val = spark.conf.get(key, "")
        if val:
            return val.strip().upper() if key != "RUN_LAYER" else val.strip().upper()
    except Exception:
        pass

    # Fall back to dbutils.widgets (job / manual notebook run)
    try:
        dbutils.widgets.text(key, default)
        val = dbutils.widgets.get(key).strip()
        if val:
            return val.upper()
    except Exception:
        pass

    return default.upper() if default else ""

group_id  = get_param("GROUP_ID",  "")
run_layer = get_param("RUN_LAYER", "ALL")

if not group_id:
    raise Exception(
        "❌  GROUP_ID is empty.\n"
        "    For DLT pipeline : set GROUP_ID in pipeline Configuration tab.\n"
        "    For Job run      : pass GROUP_ID as a job parameter.\n"
        "    For manual run   : set GROUP_ID widget at the top of the notebook."
    )

# Auto-detect layer from GROUP_ID suffix
if run_layer in ("ALL", ""):
    if   group_id.endswith("_L0"): run_layer = "L0"
    elif group_id.endswith("_L1"): run_layer = "L1"
    elif group_id.endswith("_L2"): run_layer = "L2"
    else: run_layer = "ALL"

print(f"{'═'*55}")
print(f"  VIEW PIPELINE")
print(f"  GROUP_ID  : {group_id}")
print(f"  RUN_LAYER : {run_layer}")
print(f"{'═'*55}")

In [0]:
# ── Auto-detect catalog ───────────────────────────────────────

_PREFERRED = "demo_catalog"
try:
    available = [r[0] for r in spark.sql("SHOW CATALOGS").collect()]
    if _PREFERRED in available:
        CATALOG = _PREFERRED
    elif "hive_metastore" in available:
        CATALOG = "hive_metastore"
        print(f"⚠️  '{_PREFERRED}' not found → using '{CATALOG}'")
    else:
        CATALOG = available[0] if available else _PREFERRED
except Exception:
    CATALOG = "hive_metastore"

print(f"  CATALOG   : {CATALOG}\n")

In [0]:
import re
import traceback
from datetime import datetime

In [0]:
# ── Helpers ───────────────────────────────────────────────────

def write_audit(table_name, layer, status, message, start_time, end_time):
    try:
        safe = str(message).replace("'", "''")[:500]
        spark.sql(f"""
            INSERT INTO {CATALOG}.admin.audit_log
            VALUES (
                '{group_id}', '{table_name}', '{layer}', '{status}',
                0, '{safe}',
                '{start_time.strftime('%Y-%m-%d %H:%M:%S')}',
                '{end_time.strftime('%Y-%m-%d %H:%M:%S')}',
                current_timestamp()
            )
        """)
    except Exception as e:
        print(f"   ⚠️  Audit write failed (non-fatal): {e}")


def extract_select(query, view_name):
    """
    Handle two formats in TRANSFORMATION_QUERY:
      Format A: pure SELECT ...
      Format B: CREATE OR REPLACE VIEW catalog.schema.vw_name AS SELECT ...
    Always returns just the SELECT part so we control the view name.
    """
    q = query.strip()

    # Strip CREATE OR REPLACE VIEW ... AS prefix if present
    pattern = re.compile(
        r'^\s*CREATE\s+OR\s+REPLACE\s+VIEW\s+[\w.`]+\s+AS\s*',
        re.IGNORECASE | re.DOTALL
    )
    q = pattern.sub('', q).strip()

    if not q.upper().startswith("SELECT"):
        raise ValueError(
            f"TRANSFORMATION_QUERY for {view_name} does not contain a SELECT.\n"
            f"    Got : {q[:100]}\n"
            f"    Fix : Store only the SELECT statement in TRANSFORMATION_QUERY,\n"
            f"          OR store CREATE OR REPLACE VIEW ... AS SELECT ..."
        )
    return q


def resolve_refs(query, src_schema, src_table):
    """Add catalog prefix to bare schema references."""
    q = query
    if src_schema and f"{src_schema}." in q and f"{CATALOG}.{src_schema}." not in q:
        q = q.replace(f"{src_schema}.", f"{CATALOG}.{src_schema}.")
    if src_table:
        for kw in ("FROM ", "JOIN "):
            bare = f"{kw}{src_table}"
            full = f"{kw}{CATALOG}.{src_schema}.{src_table}"
            if bare in q and full not in q:
                q = q.replace(bare, full)
    return q


def create_view(view_name, select_sql, tgt_schema):
    """Create or replace a view in the target schema."""
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{tgt_schema}")
    full_view = f"{CATALOG}.{tgt_schema}.{view_name}"
    spark.sql(f"CREATE OR REPLACE VIEW {full_view} AS {select_sql}")

    # Verify it was created
    spark.sql(f"SELECT 1 FROM {full_view} LIMIT 1")
    print(f"   ✅ View ready : {full_view}")
    return full_view


In [0]:
# ── Core: process one layer's VIEW rows ───────────────────────

def process_view_layer(detail_table, layer_name):
    print(f"\n{'─'*55}")
    print(f"  {layer_name} — VIEW CREATION")
    print(f"{'─'*55}")

    try:
        rows = spark.sql(f"""
            SELECT SOURCE_OBJ_SCHEMA, SOURCE_OBJ_NAME,
                   TARGET_SCHEMA, TARGET_TABLE,
                   LOAD_TYPE, TRANSFORMATION_QUERY
            FROM {CATALOG}.admin.{detail_table}
            WHERE DATA_FLOW_GROUP_ID = '{group_id}'
            AND   IS_ACTIVE           = 'Y'
            AND   UPPER(LOAD_TYPE)    = 'VIEW'
        """).collect()
    except Exception as e:
        raise RuntimeError(
            f"❌  Cannot read {detail_table}.\n"
            f"    Detail : {e}\n"
            f"    Fix    : Run deployment pipeline to create {detail_table}."
        )

    if not rows:
        print(f"  ℹ️  No VIEW rows in {detail_table} for '{group_id}' — skipping.")
        return

    print(f"  Views to create : {len(rows)}")
    all_ok = True

    for row in rows:
        src_schema  = row["SOURCE_OBJ_SCHEMA"] or ""
        src_table   = row["SOURCE_OBJ_NAME"]   or ""
        tgt_schema  = row["TARGET_SCHEMA"]      or "silver"
        tgt_table   = row["TARGET_TABLE"]       or ""
        trans_query = row["TRANSFORMATION_QUERY"] or ""

        # View name: prefer vw_ prefix convention
        view_name = tgt_table if tgt_table.startswith("vw_") else f"vw_{tgt_table}"

        print(f"\n  ▶  {CATALOG}.{src_schema}.{src_table}")
        print(f"     → VIEW : {CATALOG}.{tgt_schema}.{view_name}")

        start  = datetime.now()
        status = "FAILED"
        msg    = ""

        try:
            if not trans_query:
                raise ValueError(
                    f"TRANSFORMATION_QUERY is empty for {view_name}.\n"
                    f"    Fix : Add SELECT statement in TRANSFORMATION_QUERY\n"
                    f"          in {detail_table} for GROUP_ID='{group_id}'."
                )

            # Extract pure SELECT
            select_sql = extract_select(trans_query, view_name)

            # Resolve catalog references
            select_sql = resolve_refs(select_sql, src_schema, src_table)

            # Create the view
            full_view  = create_view(view_name, select_sql, tgt_schema)
            status     = "SUCCESS"
            msg        = f"View created: {full_view}"

        except Exception as e:
            all_ok = False
            msg    = (
                f"\n  ┌─ ERROR in {layer_name}/{view_name} {'─'*30}\n"
                f"  │  Type    : {type(e).__name__}\n"
                f"  │  Message : {str(e)[:300]}\n"
                f"  └{'─'*50}"
            )
            print(msg)

        finally:
            end = datetime.now()
            print(f"   ⏱  Duration : {round((end-start).total_seconds(),1)}s")
            write_audit(view_name, layer_name, status, msg, start, end)

    if not all_ok:
        raise Exception(
            f"❌  {layer_name} VIEW creation FAILED for '{group_id}'.\n"
            f"    Check audit: SELECT * FROM {CATALOG}.admin.audit_log\n"
            f"                 WHERE DATA_FLOW_GROUP_ID='{group_id}'\n"
            f"                 ORDER BY LOAD_TS DESC"
        )

In [0]:
# ── Execute ───────────────────────────────────────────────────

pipeline_start = datetime.now()

if run_layer in ("ALL", "L1"):
    process_view_layer("data_flow_l1_detail", "L1")

if run_layer in ("ALL", "L2"):
    process_view_layer("data_flow_l2_detail", "L2")

In [0]:
# ── Summary ───────────────────────────────────────────────────

total = round((datetime.now() - pipeline_start).total_seconds(), 1)

print(f"\n{'═'*55}")
print(f"  ✅  VIEW PIPELINE COMPLETE")
print(f"  GROUP_ID  : {group_id}")
print(f"  LAYER     : {run_layer}")
print(f"  CATALOG   : {CATALOG}")
print(f"  TOTAL TIME: {total}s")
print(f"{'─'*55}")
print(f"  Check created views:")
print(f"  SHOW VIEWS IN {CATALOG}.silver;")
print(f"  SHOW VIEWS IN {CATALOG}.gold;")
print(f"{'═'*55}")